In [87]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv") # 훈련을 위한 데이터
test = pd.read_csv("test.csv")  #예측을 위한 데이터

# train.head()
# train.info()
# test.info()

# 결측치 탐색
train.isnull().sum()

# 결측치 대체 - age(숫자) - 중앙값(median)
median1 = train['age'].median()
# print(median1)
median2 = train['age'].median()

train['age'] = train['age'].fillna(median1)
train['hours.per.week'] = train['hours.per.week'].fillna(median2)
test['age'] = test['age'].fillna(median1)
test['hours.per.week'] = test['hours.per.week'].fillna(median2)

# print(train.isnull().sum())
test.isnull().sum()

# 결측치 대체 - 문자형 -> 최빈값(mode)
mode1 = train['workclass'].mode()[0]
# print(mode1)
mode2 = train['occupation'].mode()[0]
mode3 = train['native.country'].mode()[0]
train['workclass'] = train['workclass'].fillna(mode1)
train['occupation'] = train['occupation'].fillna(mode2)
train['native.country'] = train['native.country'].fillna(mode3)

test['workclass'] = test['workclass'].fillna(mode1)
test['occupation'] = test['occupation'].fillna(mode2)
test['native.country'] = test['native.country'].fillna(mode3)

# print(train.isnull().sum())
# print(test.isnull().sum())

# 데이터 선택(X-독립변수, y-종속변수)
target = train.pop("income")
# train = train.drop(columns=['income'])
# target = train['income']

# 인코딩 - 문자형 -> 숫자형으로 변환
# 라벨(Label Encoding)
encoder = LabelEncoder()

# train.info()
cols = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 
        'race', 'sex', 'native.country']

for col in cols:
    train[col] = encoder.fit_transform(train[col])
    test[col] = encoder.fit_transform(test[col])

# print(train.info())
# print(test.info())

# 데이터 분할(훈련/검증)
# train_test_split?
X_train, X_test, y_train, y_test = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# 모델 생성 및 학습, 평가
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=333)
model.fit(X_train, y_train)
# 예측(검증용 데이터)
pred1 = model.predict_proba(X_test)
# print(pred1)

# 성능 평가 - roc_auc_score
# '> 50K'의 확률
from sklearn.metrics import roc_auc_score

score = roc_auc_score(y_test, pred1[:, 1])
print("score:", score)

# test 데이터로 예측(최종)
pred2 = model.predict_proba(test)
# print(pred2)

# 파일 제출
df = pd.DataFrame({
    "pred": pred2[:, 1]
})

# 파일 제출
df.to_csv("result.csv", index=False)

# 읽기
pd.read_csv("result.csv")

(23443, 15) (5861, 15) (23443,) (5861,)
score: 0.91125469722388


,pred
0,0.03
1,0.00
2,0.06
3,0.72
4,0.07
...,...
3252,0.00
3253,0.48
3254,0.11
3255,0.01
